# 모기업·계열사·종속기업 통합 데이터 명세서

## 1. 데이터셋 개요

| 항목 | 내용 |
| --- | --- |
| 파일명 | `data/clean/모기업_계열사_종속기업_통합.csv` |
| 데이터 목적 | 모기업, 계열회사, 종속기업 사이의 연결 관계와 각 기업의 지역·주소·업종 정보를 하나의 관계 테이블로 제공 |
| 데이터 단위 | 한 행은 `case`가 정의하는 하나의 기업 관계를 나타냄 |
| 크기 | **17,398행 × 17열** |
| 기준 기간 | 원본 CSV에 기준일·기준연도 컬럼이 없어 **알 수 없음** |
| 관계 구분 | `case=1`: 모기업→계열사, `case=2`: 모기업→종속기업, `case=3`: 모기업→계열사→종속기업 |
| 중복 제거 기준 | `case`, `top_crno`, `affiliate_crno`, `subsidiary_name`의 조합 |
| 주요 식별자 | 모기업·계열사는 13자리 법인등록번호(`top_crno`, `affiliate_crno`)를 사용하며, 종속기업은 별도 법인등록번호 없이 이름과 주소 등을 사용 |
| 관심 변수 | 관계 유형(`case`), 기업 식별자와 기업명, 지역·주소·업종, 종속기업 국내 여부와 정규화 이름 |
| 파일 내 중복 | 완전히 동일한 행 0건, 중복 제거 기준의 중복 0건 |

## 2. 행 구조와 `case` 정의

`case`에 따라 한 행이 표현하는 관계와 사용되는 컬럼 묶음이 달라진다. 따라서 구조적으로 사용하지 않는 컬럼의 공란과 실제 정보 누락을 구분해야 한다.

| case | 관계 경로 | 행 수 | 비율 | 반드시 사용되는 관계 컬럼 | 구조적으로 비어 있는 컬럼 묶음 |
| ---: | --- | ---: | ---: | --- | --- |
| 1 | 모기업 → 계열회사 | 9,513 | 54.68% | `top_*`, `affiliate_*` | 모든 `subsidiary_*`, `domestic`, `name_norm` |
| 2 | 모기업 → 종속기업 | 1,831 | 10.52% | `top_*`, `subsidiary_*`, `domestic`, `name_norm` | 모든 `affiliate_*` |
| 3 | 모기업 → 계열회사 → 종속기업 | 6,054 | 34.80% | `top_*`, `affiliate_*`, `subsidiary_*`, `domestic`, `name_norm` | 없음. 다만 원천 정보 부족으로 일부 값은 결측 |

### 관계 유형별 실제 예시

아래 예시는 관계 구조를 보여 주기 위해 핵심 컬럼만 표시했다.

| case | top_crno / top_corpNm | affiliate_crno / affiliate_corpNm | subsidiary_name | domestic | 해석 |
| ---: | --- | --- | --- | --- | --- |
| 1 | `1101110000086` / 롯데쇼핑(주) | `1101110014764` / 롯데건설주식회사 | 비어 있음 | 비어 있음 | 롯데쇼핑(주)의 계열회사로 롯데건설주식회사가 연결됨 |
| 2 | `1101110019962` / 현대차증권(주) | 비어 있음 | 씨더블유한마음제일차㈜ | 국내 | 계열사 단계를 거치지 않고 모기업과 종속기업이 직접 연결됨 |
| 3 | 복수 법인등록번호 / 복수 모기업명 | `1101110017867` / 한솔홀딩스(주) | 한솔페이퍼텍(주) | 국내 | 여러 최상위 모기업 후보가 한솔홀딩스(주)를 거쳐 종속기업과 연결됨 |

## 3. 컬럼 명세

CSV는 식별자의 앞자리 0과 복수값 표현을 보존하기 위해 분석 시 전체 컬럼을 문자열로 읽는 것이 안전하다. `case`만 1·2·3의 범주형 값으로 변환할 수 있다.

| 컬럼명 | 권장 자료형 | 결측 건수 | 결측률 | 비결측 고유값 수 | 설명 및 사용 시 주의사항 |
| --- | --- | ---: | ---: | ---: | --- |
| `case` | 범주형 정수 | 0 | 0.00% | 3 | 관계 유형. 값은 1, 2, 3이며 상세 의미는 위의 case 정의 참고 |
| `top_crno` | 문자열 | 0 | 0.00% | 2,118 | 최상위 모기업 법인등록번호. 개별 번호는 13자리 숫자 문자열이며, `case=3`에서는 여러 번호가 `;`로 연결될 수 있음 |
| `top_corpNm` | 문자열 | 0 | 0.00% | 2,112 | 최상위 모기업명. 복수 모기업이면 `top_crno`와 같은 순서로 `;` 연결 |
| `top_region` | 문자열 | 9,340 | 53.68% | 35 | 모기업 주소에서 추출한 시·도 축약명. 복수 모기업이면 `;` 연결 |
| `top_addr` | 문자열 | 9,340 | 53.68% | 652 | 모기업 주소. 복수 모기업이면 `;` 연결 |
| `top_sicNm` | 문자열 | 11,134 | 64.00% | 306 | 모기업 업종명. 기업개요 원천의 `sicNm`, 없으면 `enpMainBizNm`을 활용한 값 |
| `affiliate_crno` | 문자열 | 1,831 | 10.52% | 1,437 | 계열회사 법인등록번호. `case=2`에서는 구조적으로 비어 있음 |
| `affiliate_corpNm` | 문자열 | 1,831 | 10.52% | 1,433 | 계열회사명. `case=2`에서는 구조적으로 비어 있음 |
| `affiliate_region` | 문자열 | 12,639 | 72.65% | 12 | 계열회사 주소에서 추출한 시·도 축약명 |
| `affiliate_addr` | 문자열 | 12,639 | 72.65% | 268 | 계열회사 주소 |
| `affiliate_sicNm` | 문자열 | 13,042 | 74.96% | 134 | 계열회사 업종명 |
| `subsidiary_name` | 문자열 | 9,513 | 54.68% | 6,963 | 종속기업 원문 이름. `case=1`에서는 구조적으로 비어 있음 |
| `subsidiary_region` | 문자열 | 11,223 | 64.51% | 17 | 종속기업 주소에서 추출한 시·도 축약명 |
| `subsidiary_addr` | 문자열 | 9,578 | 55.05% | 3,534 | 종속기업 주소. 원본의 상동 표기를 같은 모기업 범위에서 앞 행 값으로 보완한 결과 |
| `subsidiary_bizCtt` | 문자열 | 9,564 | 54.97% | 2,527 | 종속기업 주요 사업 내용 |
| `domestic` | 범주형 문자열 | 9,513 | 54.68% | 2 | 종속기업 국내 여부. 비결측 값은 `국내`, `모름` |
| `name_norm` | 문자열 | 9,513 | 54.68% | 6,701 | 종속기업명에서 법인 표기·공백·특수문자 등을 정리한 매칭용 정규화 이름 |

## 4. 결측값 해석

전체 결측률에는 관계 유형상 사용하지 않는 컬럼의 구조적 공란이 포함된다. 실제 품질을 판단할 때는 해당 컬럼을 사용하는 case 안에서 다시 확인해야 한다.

| 대상 case | 컬럼 | case 내부 결측 건수 | case 내부 결측률 | 해석 |
| ---: | --- | ---: | ---: | --- |
| 1 | `top_region`, `top_addr` | 각 7,556 | 79.43% | 모기업 주소 원천값 부족 |
| 1 | `top_sicNm` | 7,818 | 82.18% | 모기업 업종 원천값 부족 |
| 1 | `affiliate_region`, `affiliate_addr` | 각 7,561 | 79.48% | 계열회사 주소 원천값 부족 |
| 1 | `affiliate_sicNm` | 7,823 | 82.23% | 계열회사 업종 원천값 부족 |
| 2 | `top_sicNm` | 961 | 52.48% | 직접 모기업의 업종 원천값 부족 |
| 2 | `subsidiary_region` | 181 | 9.89% | 주소가 있어도 시·도를 추출하지 못한 값 포함 |
| 2 | `subsidiary_addr` | 37 | 2.02% | 종속기업 주소 원천값 부족 |
| 2 | `subsidiary_bizCtt` | 20 | 1.09% | 종속기업 사업 내용 원천값 부족 |
| 3 | `top_region`, `top_addr` | 각 1,784 | 29.47% | 복수 최상위 모기업 후보의 주소 원천값 부족 |
| 3 | `top_sicNm` | 2,355 | 38.90% | 복수 최상위 모기업 후보의 업종 원천값 부족 |
| 3 | `affiliate_region`, `affiliate_addr` | 각 3,247 | 53.63% | 중간 계열회사의 주소 원천값 부족 |
| 3 | `affiliate_sicNm` | 3,388 | 55.96% | 중간 계열회사의 업종 원천값 부족 |
| 3 | `subsidiary_region` | 1,529 | 25.26% | 주소가 있어도 시·도를 추출하지 못한 값 포함 |
| 3 | `subsidiary_addr` | 28 | 0.46% | 종속기업 주소 원천값 부족 |
| 3 | `subsidiary_bizCtt` | 31 | 0.51% | 종속기업 사업 내용 원천값 부족 |

## 5. 값 표현 및 데이터 품질 규칙

| 점검 항목 | 결과 | 활용 시 유의사항 |
| --- | --- | --- |
| 완전 중복 행 | 0건 | 현재 파일에는 모든 컬럼이 동일한 행이 없음 |
| 관계 중복 | 0건 | 생성 단계의 복합 기준(`case`, `top_crno`, `affiliate_crno`, `subsidiary_name`)으로 중복 제거됨 |
| 법인등록번호 형식 | 분리된 모든 `top_crno`, `affiliate_crno`가 13자리 숫자 | 숫자형으로 변환하지 말고 문자열로 유지 |
| 복수 모기업 행 | 5,208건 | 전체의 29.93%, `case=3`의 86.02%. `top_*` 계열 값을 `;`로 함께 분리해야 함 |
| `domestic` 분포 | 국내 6,135건, 모름 1,750건 | 빈 값 9,513건은 모두 `case=1`의 구조적 공란 |
| 종속기업 식별 | 법인등록번호 없음 | 우선 `name_norm`과 `subsidiary_addr`를 함께 사용하고, 주소가 없으면 사업 내용·국내 여부 등을 보조 정보로 사용 |
| 고유값 수 해석 | `top_crno` 문자열 2,118개, `;` 분리 후 개별 법인등록번호 1,724개 | 복수값 문자열 자체를 하나의 회사 식별자로 집계하면 안 됨 |
| 순서 대응 | 복수값 `top_crno`, `top_corpNm`, `top_region`, `top_addr`, `top_sicNm`은 같은 순서로 연결 | 컬럼별로 독립 정렬하면 기업 속성 간 대응이 깨질 수 있음 |

## 6. 분석 및 그래프 적재 권장 방식

| 작업 | 권장 방식 |
| --- | --- |
| CSV 로드 | 법인등록번호와 복수값을 보존하도록 `dtype=str`, `encoding="utf-8-sig"` 사용 |
| 모기업 전처리 | `top_crno`에 `;`가 있으면 관련 `top_*` 컬럼을 동일한 위치 기준으로 함께 분해 |
| 관계 생성 | `case=1`은 모기업→계열사, `case=2`는 모기업→종속기업, `case=3`은 각 모기업→계열사와 계열사→종속기업 관계로 분리 |
| 모기업·계열사 노드 키 | 법인등록번호 사용 |
| 종속기업 노드 키 | `name_norm + subsidiary_addr` 조합을 우선 사용. 주소가 없을 때의 대체 키는 중복 가능성을 별도로 검토 |
| 결측 처리 | 구조적 공란은 유지하고, 실제 원천값 누락을 0이나 `모름`으로 일괄 치환하지 않음 |
| 기간 필터 | 기준 기간 컬럼이 없으므로 이 파일만으로 시점별 비교를 수행하지 않음 |
